# BRCA1 pre-neoplastic comparative analysis: do rescued "noise" cells change LP abundance?

Pal et al. (2021) compared 8 WT (normal premenopausal) and 4 BRCA1 pre-neoplastic "Total" samples
after standard QC (Fig 4A–C, Appendix S1A–C). The **null** reproduction of that analysis lives in
[`brca1-preneoplastic-null.ipynb`](brca1-preneoplastic-null.ipynb). This notebook asks whether the
luminal progenitor (LP) fraction changes between genotypes when cells discarded by QC are rescued
using the PBS classes, and runs the same analysis for each PBS signal and for combinations of them.

| Step | Question | Status |
|---|---|---|
| 0 | Are WT and BRCA1 samples comparable in depth, genes and mito %? | **this notebook** |
| 0b | Minimum detectable effect from WT donors only (blind); hypothesis test or estimation? | [`brca1-preneoplastic-design-sensitivity.ipynb`](brca1-preneoplastic-design-sensitivity.ipynb) |
| 1 | Classify noise with frozen WT quantiles; build Arms B, C, D | next |
| 2 | Assign identity by mapping every cell onto Arm B clusters | |
| 3 | Drop rescued LP calls that are not ≥2× enriched over the sample's ambient profile | |
| 4 | LP / (LP + basal + ML) per donor | |
| 5 | Exact permutation test on per-donor Δ (Arm D − Arm B) + depth-matched, random-rescue and dissociation-stress controls; framing fixed by Step 0b's `design_decision.json` | |
| 6 | Support only if positive, survives ambient filter, survives depth matching, and exceeds random rescue | |

**Unit of analysis is the donor throughout.** Reusable code lives in
`signals_in_the_noise.analysis` (`brca1_cohort`, `sample_comparability`, `statistics`) and is
unit-tested under `tests/analysis`.

**Recommended environment:** same SageMaker instance and conda env as the null notebook. Step 0 only
needs Python; the first run loads GSE161529 once and caches a per-cell QC table.

## Setup

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

from signals_in_the_noise.analysis.brca1_cohort import (
    DONORS,
    GENOTYPE_PALETTE,
    build_cell_table,
    compare_qc_pass_counts_to_paper,
    compare_qc_pass_totals_to_paper,
    load_or_build_cell_table,
)
from signals_in_the_noise.analysis.sample_comparability import (
    DEFAULT_DEPTH_TOLERANCE,
    POPULATIONS,
    QC_METRICS,
    assess_depth_matching,
    compare_genotypes,
    summarize_donors,
)
from signals_in_the_noise.config import get_data_path
from signals_in_the_noise.utils.logging_config import setup_logging
from signals_in_the_noise.utils.visualization import (
    plot_donor_medians_by_genotype,
    plot_qc_metrics_by_donor,
)

setup_logging()

In [ ]:
# Parameters
FORCE_REBUILD_CELL_TABLE = False  # True re-reads every donor's AnnData instead of the cached table
DEPTH_TOLERANCE = DEFAULT_DEPTH_TOLERANCE  # |BRCA1 / WT - 1| above this makes Step 5 depth matching mandatory

ANALYSIS_DIR = get_data_path("GSE161529_analysis_cache") / "brca1_preneoplastic_comparative"
STEP0_DIR = ANALYSIS_DIR / "step0_sample_comparability"
STEP0_DIR.mkdir(parents=True, exist_ok=True)

CELL_TABLE_PATH = ANALYSIS_DIR / "cell_table.csv.gz"
ANNOTATIONS_PATH = get_data_path("GSE161529_annotations_df.csv")


def save_figure(fig: plt.Figure, name: str) -> None:
    fig.savefig(STEP0_DIR / f"{name}.png", dpi=200, bbox_inches="tight")


print("ANALYSIS_DIR:", ANALYSIS_DIR)
pd.DataFrame([(d.author_id, d.geo_stem, d.genotype) for d in DONORS], columns=["donor", "geo_stem", "genotype"])

## Step 0 — Are the samples comparable?

Before any genotype comparison, check that WT and BRCA1 samples were sequenced comparably. If BRCA1
samples are systematically shallower or deeper, depth alone can shift QC outcomes, PBS
classification and cell-type calls, and could masquerade as a genotype effect.

**Method.** For each donor, take the median of UMI counts, genes detected and mitochondrial % per cell.
Compare genotypes on the median of those donor medians (donor is the unit, so a donor with many cells
does not dominate), with an exact permutation test over all 495 relabelings of 4 BRCA1 vs 8 WT donors.

Three cell populations are compared:

| Population | Meaning | Why it matters |
|---|---|---|
| `qc_pass` | cells kept by the paper QC | defines Arm B and the reference clusters |
| `noise` | cells discarded by the paper QC | the pool PBS cells are rescued from |
| `all_barcodes` | everything deposited in GEO | context |

**Decision rule.** If the BRCA1 median depth differs from WT by more than ±20% in `qc_pass` **or**
`noise`, the depth-matched run in Step 5 is mandatory rather than optional.

### 0.1 Collect per-cell QC metrics

QC metrics come from the project preprocessor (`scanpy.pp.calculate_qc_metrics` on raw counts), and
`is_noise` is the paper's QC applied with the per-sample thresholds from the supplementary tables.
The first run loads GSE161529 and caches the 12-donor table; later runs read the cache.

In [ ]:
def _build_from_gse161529() -> pd.DataFrame:
    from signals_in_the_noise.preprocessing.gse161529 import GSE161529

    return build_cell_table(GSE161529().get_dataset)


cells = load_or_build_cell_table(CELL_TABLE_PATH, _build_from_gse161529, force=FORCE_REBUILD_CELL_TABLE)

cells_per_population = pd.DataFrame(
    {name: cells.loc[mask(cells)].groupby("donor", observed=True).size() for name, mask in POPULATIONS.items()}
)
cells_per_population.insert(0, "genotype", cells.groupby("donor", observed=True)["genotype"].first())
cells_per_population

### 0.2 Reproduce the paper's QC-pass counts

Interpretability with the authors depends on starting from exactly their cells. Per-donor QC-pass
counts must match `number-of-cells-after-filtering` from the supplementary table, and the genotype
totals should match the paper text (36,526 Normal + 23,240 BRCA1 = 59,766).

In [ ]:
per_donor_check = compare_qc_pass_counts_to_paper(cells, pd.read_csv(ANNOTATIONS_PATH))
display(per_donor_check)
assert per_donor_check["matches"].all(), "QC-pass counts differ from the paper; fix before interpreting Step 0."

totals_check = compare_qc_pass_totals_to_paper(cells)
display(totals_check)
if not totals_check["matches"].all():
    print("WARNING: genotype totals differ from the paper text; report this alongside Step 0 results.")

### 0.3 Per-donor distributions

One figure per population; each box is a donor (whiskers 1.5×IQR, outliers hidden), coloured with the
authors' Appendix S1A palette (blue = WT/Normal, gold = BRCA1). Counts and genes are on a log scale.

In [ ]:
for population, mask in POPULATIONS.items():
    axes = plot_qc_metrics_by_donor(
        cells.loc[mask(cells)],
        QC_METRICS,
        palette=GENOTYPE_PALETTE,
        title=f"Per-donor QC distributions: {population}",
    )
    save_figure(axes[0].figure, f"qc_by_donor_{population}")
    plt.show()

### 0.4 Genotype comparison on donor medians

Each point below is one donor's median; the black bar is the genotype median of donor medians.
`ratio` is BRCA1 / WT; `permutation_p` is the two-sided exact permutation p-value on donor medians
(minimum achievable ≈ 0.002). With 4 vs 8 donors, a large p-value is weak evidence of equality, so
the decision relies on the size of the difference, not on significance.

In [ ]:
donor_summary = summarize_donors(cells)
genotype_comparison = compare_genotypes(donor_summary)

donor_summary.to_csv(STEP0_DIR / "donor_qc_summary.csv", index=False)
genotype_comparison.to_csv(STEP0_DIR / "genotype_qc_comparison.csv", index=False)

genotype_comparison.style.format(
    {"WT_median": "{:,.1f}", "BRCA1_median": "{:,.1f}", "ratio": "{:.2f}", "relative_difference": "{:+.1%}", "permutation_p": "{:.3f}"}
)

In [ ]:
axes = plot_donor_medians_by_genotype(donor_summary, QC_METRICS, palette=GENOTYPE_PALETTE, populations=POPULATIONS)
axes[0].figure.suptitle("Donor medians by genotype", fontsize=16)
axes[0].figure.tight_layout()
save_figure(axes[0].figure, "donor_medians_by_genotype")
plt.show()

In [ ]:
# One row per donor: median of each metric in each population (the points in the figure above).
donor_medians = donor_summary.pivot_table(
    index=["genotype", "donor"], columns=["population", "metric"], values="median", observed=True
).reindex(columns=list(POPULATIONS), level="population").round(1)
donor_medians.to_csv(STEP0_DIR / "donor_medians_wide.csv")
donor_medians

### 0.5 Decision: is the depth-matched run in Step 5 mandatory?

In [ ]:
depth_decision = assess_depth_matching(genotype_comparison, tolerance=DEPTH_TOLERANCE)
depth_decision.to_json(STEP0_DIR / "depth_matching_decision.json")
print(depth_decision.summary())

The decision is written to `step0_sample_comparability/depth_matching_decision.json`; Step 5
reads it with `DepthMatchingDecision.from_json` so the depth-matched control cannot be skipped
silently when it is mandatory.

**Report back after running:** the `per_donor_check` / `totals_check` tables, the
`genotype_comparison` table, the decision line above, and the donor-medians figure.